# Pathlib Module

## Quick notes

### What this covers
`pathlib.Path` — the modern, object-oriented way to work with filesystem paths, replacing string-based `os.path`.

### Key points
- **`Path("a") / "b"`** — join paths with `/` instead of `os.path.join()`.
- **Components** — `.name`, `.stem` (name without suffix), `.suffix`, `.parent`, `.parts`.
- **Checks** — `.exists()`, `.is_dir()`, `.is_file()`, `.is_absolute()`.
- **Filesystem ops** — `.mkdir(exist_ok=True)`, `.rmdir()`, `.write_text()` / `.read_text()`, `.unlink()` (delete file).
- **`.iterdir()`** — list a directory's direct contents. **`.glob(pattern)`** — pattern-match files; needs explicit `**` to recurse (`glob("**/*.py")`). **`.rglob(pattern)`** — shorthand that's always recursive.
- **`.resolve()`** — get the absolute, fully resolved path.
- **`.with_suffix(".json")`** / **`.with_name("new.txt")`** — return a new `Path` with just that piece changed.
- **`.stat()`** — file metadata: `.st_size`, `.st_mtime` (modification time, as a timestamp).
- **`PurePosixPath` / `PureWindowsPath`** — pure string manipulation for paths from a *different* OS, no filesystem access. `Path` itself is OS-specific and *does* touch the real filesystem.
- **`Path.home()`** / **`Path("~/data").expanduser()`** — resolve the user's home directory / `~`.

### Gotchas (the part worth revising twice)
- `glob("*.py")` only searches the current directory — you need `glob("**/*.py")` (with `**`) to recurse. `rglob("*.py")` is the recursive shortcut, equivalent to `glob("**/*.py")`.
- `PurePath` types never touch disk (safe to build paths for an OS you're not running), while `Path` is concrete and OS-specific — don't call `.exists()` on a `PureWindowsPath` from Linux, it's not meant to.
- `.with_suffix()`/`.with_name()` return a *new* `Path` — they don't mutate the original.
- Prefer `pathlib` over `os.path` in new code — same capability, more readable (`Path('a') / 'b'` vs `os.path.join('a', 'b')`).

### When it matters
Any code touching the filesystem — reading/writing files, walking directories, building cross-platform paths, or inspecting file metadata.

### One-line recap example
```python
from pathlib import Path
p = Path("data") / "report.txt"
p.parent.mkdir(exist_ok=True)
p.write_text("hello")          # object-oriented, no os.path.join needed
```

In [1]:
# notebooks have no __file__; point it at this notebook
__file__ = str(__import__("pathlib").Path("05-Pathlib-Module.ipynb").resolve())

In [2]:
from pathlib import Path

## 01. pathlib - Introduction

- Object-oriented filesystem paths, a modern alternative to os.path.
- Path objects support / for joining and many convenient methods.

## 02. Creating Path Objects

In [3]:
current = Path(".")
print("\n# 02. Creating Path Objects")
print("current:", current.resolve())


# 02. Creating Path Objects
current: <repo>/Python/06-Standard Library


## 03. Joining Paths with / (instead of os.path.join)

In [4]:
sample = Path("folder") / "subfolder" / "file.txt"
print("\n# 03. Joining Paths with /")
print(sample)


# 03. Joining Paths with /
folder/subfolder/file.txt


## 04. Path Components

In [5]:
print("name:", sample.name)
print("stem (name w/o suffix):", sample.stem)
print("suffix:", sample.suffix)
print("parent:", sample.parent)
print("parts:", sample.parts)

name: file.txt
stem (name w/o suffix): file
suffix: .txt
parent: folder/subfolder
parts: ('folder', 'subfolder', 'file.txt')


## 05. Checking Existence and Type

In [6]:
print("exists:", current.exists())
print("is_dir:", current.is_dir())
print("is_file:", current.is_file())

exists: True
is_dir: True
is_file: False


## 06. Creating and Removing Directories/Files

In [7]:
demo_dir = Path("temp_pathlib_dir")
demo_dir.mkdir(exist_ok=True)
print("\n# 06. Creating and Removing Directories/Files")
print("created:", demo_dir.exists())

demo_file = demo_dir / "note.txt"
demo_file.write_text("Hello from pathlib!")
print("file content:", demo_file.read_text())

demo_file.unlink()   # delete file
demo_dir.rmdir()     # delete (now empty) directory
print("cleaned up, exists:", demo_dir.exists())


# 06. Creating and Removing Directories/Files
created: True
file content: Hello from pathlib!
cleaned up, exists: False


## 07. Iterating Directory Contents

In [8]:
for item in list(current.iterdir())[:5]:
    print(item)

01-Collections-Module.ipynb
22-Networking-Basics.ipynb
18-Text-Utilities.ipynb
13-Time-Module.ipynb
15-Hashing-And-Encoding.ipynb


## 08. Pattern Matching with glob()

In [9]:
py_files = list(Path("..").glob("**/*.py"))
print(f"found {len(py_files)} .py files (searched recursively)")

found 104 .py files (searched recursively)


## 09. Absolute vs Relative Paths

In [10]:
print("is_absolute:", sample.is_absolute())
print("absolute form:", sample.resolve())

is_absolute: False
absolute form: <repo>/Python/06-Standard Library/folder/subfolder/file.txt


## 10. pathlib vs os.path (comparison)

## 11. rglob() vs glob() - Recursive vs Single-Level

- glob('*.py') only looks in the current directory.
- glob('**/*.py') needs the explicit '**' to recurse.
- rglob('*.py') is shorthand for glob('**/*.py') - always recursive.

In [11]:
top_level_only = list(current.glob("*.py"))
recursive_manual = list(Path("..").glob("**/*.py"))
recursive_rglob = list(Path("..").rglob("*.py"))
print("glob('*.py') in current dir:", len(top_level_only), "files")
print("glob('**/*.py') count:", len(recursive_manual))
print("rglob('*.py') count:   ", len(recursive_rglob), "(same as above)")

glob('*.py') in current dir: 0 files
glob('**/*.py') count: 104
rglob('*.py') count:    104 (same as above)


## 12. with_suffix() and with_name() - Path Transformation

In [12]:
original = Path("reports/summary.txt")
print("original:", original)
print("with_suffix('.json'):", original.with_suffix(".json"))
print("with_name('final.txt'):", original.with_name("final.txt"))

original: reports/summary.txt
with_suffix('.json'): reports/summary.json
with_name('final.txt'): reports/final.txt


## 13. Path.stat() - File Metadata

In [13]:
from datetime import datetime

this_file = Path(__file__)
info = this_file.stat()
print("size (bytes):", info.st_size)
print("last modified:", datetime.fromtimestamp(info.st_mtime).strftime("%Y-%m-%d %H:%M:%S"))

size (bytes): 14303
last modified: 2026-11-17 00:34:16


## 14. PurePath vs Path - Comparing Paths Across Platforms

- PurePath (PurePosixPath/PureWindowsPath) does pure string manipulation,
  no filesystem access - works even for paths from a different OS.
- Path (the concrete class used throughout this file) can also touch the
  real filesystem (exists(), read_text(), etc.) on the current OS.

In [14]:
from pathlib import PurePosixPath, PureWindowsPath

print("\n# 14. PurePath vs Path")
posix_style = PurePosixPath("/home/user/file.txt")
windows_style = PureWindowsPath(r"C:\Users\user\file.txt")
print("posix parts:", posix_style.parts)
print("windows parts:", windows_style.parts)
print("# PurePath never touches disk; Path does and is OS-specific.")


# 14. PurePath vs Path
posix parts: ('/', 'home', 'user', 'file.txt')
windows parts: ('C:\\', 'Users', 'user', 'file.txt')
# PurePath never touches disk; Path does and is OS-specific.


## 15. expanduser() and Path.home()

In [15]:
print("home():", Path.home())
print("expanduser('~/data'):", Path("~/data").expanduser())

home(): ~
expanduser('~/data'): ~/data
